In [1]:
from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras    
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator
# https://github.com/sieu-n/KoOCR-tensorflow/blob/main/utils/model_architectures.py

2024-06-10 12:15:58.749358: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-10 12:15:59.484192: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
#PRE DEFINE

DATA_SIZE = 10000
VALID_DATA_SIZE = DATA_SIZE / 5

ORG_TRAIN_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

ORG_VALID_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

SAVE_DIR = "/root/Data/hangul/weights"
WEIGHT_FILE = SAVE_DIR + "/remodelA_1.weights.h5"
KERAS_FILE = SAVE_DIR + "/remodelA_1.keras"

In [3]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-06-10 12:16:00.217709: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.237467: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.237512: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.423777: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.423836: I external/local_xla/xla/stream_executor

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 15572094606170290751
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 2083518294787448068
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [4]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18,
'0':19, '1':20, '2':21, '3':22, '4':23, '5':24, '6':25, '7':26, '8':27, '9':28, '-':29}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20, None:21}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ',
            19: '0', 20:'1', 21:'2', 22:'3', 23:'4', 24:'5', 25:'6', 26:'7', 27:'8', 28:'9', 29:'-'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ', 21:None}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [5]:

def getSpecificExtensionFiles(path, extension):
    out = []
    for (path, dir, files) in os.walk(path):
        for filename in files:
            ext = os.path.splitext(filename)[-1]
            if ext == extension:
                #print("%s/%s" % (path, filename))
                out.append(path + "/" + filename)
    return out

In [6]:
import matplotlib.pyplot as plt
import random

# draw_text = '람'
# font = "/root/Data/font/clova-all/가람연꽃/나눔손글씨_가람연꽃.ttf"

# fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")

def DataAugmentation():
    augment = tf.keras.Sequential([
        tf.keras.layers.experimental.preprocessing.RandomZoom( height_factor=(-0.2, 0.1),width_factor=(-0.2, 0.1),fill_mode='constant'),
        tf.keras.layers.experimental.preprocessing.RandomRotation(0.1,fill_mode='constant'),
        tf.keras.layers.experimental.preprocessing.RandomTranslation(0.1,0.1,fill_mode='constant')
        
    ])
    return augment

from PIL import Image,ImageDraw,ImageFont

def CreateFontImage(str, fontPath):
    font = ImageFont.truetype(fontPath, 28, encoding = 'utf-8')
    left, top, right, bottom = font.getbbox(str)
    width = right - left
    height = bottom - top
    
    canvas = Image.new('RGB', (width + 10, height + 14), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((3,3), str, 'black', font)
    
    #print(canvas)
    img = tf.image.convert_image_dtype(canvas, tf.float32)
    img = tf.image.resize(img, (64, 64))    
    img = np.array(img)
    img = np.expand_dims(img, axis=0)
    
    #plt.imshow(img)
    #print(img)

    # save the blank canvas to a file
    #canvas.save("unicode-text.png", "PNG")
    #canvas.show()
    return img

    
#CreateFontImage(draw_text, font)

def getFontImage(fontPath, imageNum):
    #fontFiles = getSpecificExtensionFiles(fontPath, ".ttf")
    
    #for i in range(1, imageNum):
        #fontidx = random.randrange(0, len(fontFiles) + 1)
    cho = random.randrange(0, 19)
    jung = random.randrange(0, 21)
    jong = random.randrange(0, 28)
    
    ja = label2ja[cho]
    mo = label2mo[jung]
    ba = label2ba[jong]

    char = unicode.join_jamos_char(ja, mo ,ba)
    #print(char)
    
    label1 = np.expand_dims(np.array(cho), axis=0)
    label2 = np.expand_dims(np.array(jung), axis=0)
    label3 = np.expand_dims(np.array(jong), axis=0)
    

    #yield CreateFontImage(char, fontFiles[fontidx])
    return CreateFontImage(char, fontPath), (label1, label2, label3)

In [7]:
synthImagePath = ""
basePath = "" + "/"

def getSynthDataset():
    cnt = 0
    
    os.os.system("shuf /root/Data/hangul/dataset/tranDataset.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    csvFile = open("")
    
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        imgFile = os.path.join(basePath, imgFile)
        
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        

        
        
        

In [8]:
def get_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)
    
    #os.system("shuf /root/Data/hangul/dataset/MergedData.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    os.system("shuf /root/Data/hangul/dataset/deDup.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    
    csvFile = open("/root/Data/hangul/dataset/shuffled_tranDataset.csv", 'r', encoding='utf-8')
    #csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        
        fontidx = random.randrange(0, len(fontFiles))
        yield getFontImage(fontFiles[fontidx], 10)

        if (cnt > DATA_SIZE): break
        else                : cnt += 1

        
        
def get_valid_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)

    os.system("shuf /root/Data/hangul/dataset/MergedValidData.csv > /root/Data/hangul/dataset/shuffled_validation.csv")    
    csvFile = open("/root/Data/hangul/dataset/shuffled_validation.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        
        fontidx = random.randrange(0, len(fontFiles))
        yield getFontImage(fontFiles[fontidx], 10)

        if (cnt > VALID_DATA_SIZE): break
        else                : cnt += 1

In [9]:
dataset = tf.data.Dataset.from_generator(get_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )


validDtaset = tf.data.Dataset.from_generator(get_valid_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )

2024-06-10 12:16:00.685647: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.685758: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.685789: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.686465: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-10 12:16:00.686509: I external/local_xla/xla/stream_executor

In [10]:
REG_LAMBDA = 0.01 # 0.001 0.1 0.05
cl2_reg = tf.keras.regularizers.l2(REG_LAMBDA)

def AddSingleLayer(inputTensor, filters, kernel_size = (3,3)):
    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_regularizer=cl2_reg)(inputTensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    return x

def residual_block(input_tensor, filters):
    Node = AddSingleLayer(inputTensor=input_tensor, filters=filters)
    
    x = layers.Conv2D(filters, (3, 3), padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, Node])
    x = layers.Activation('relu')(x)
    
    return x

def CommonBranchBlock(inputTensor):
    #x = AddSingleLayer(inputTensor, 128, kernel_size=(7,7))
    x = residual_block(inputTensor, 128)
    #x = AddSingleLayer(inputTensor, 128)
    x = AddSingleLayer(x, 256)
    x = layers.SpatialDropout2D(0.2)(x)
    x = layers.MaxPooling2D(pool_size = 2)(x)
    x = AddSingleLayer(x, 512)
    x = layers.SpatialDropout2D(0.2)(x)
    x = layers.MaxPooling2D(pool_size = 2)(x)
    
    return x

def BranchBlock(inputTensor, filters, layerSize, lastLayerName):
    x = layers.Conv2D(filters * 2, (3, 3), padding='same', kernel_regularizer=cl2_reg)(inputTensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size = 2)(x)
    x = layers.SpatialDropout2D(0.3)(x)
    
    
    x = residual_block(x,filters=filters)
    #x = layers.MaxPooling2D(pool_size = 2)(x)
    
    #x = AddSingleLayer(x, filters=filters * 2, kernel_size=(3,3))
    #x = layers.MaxPooling2D(pool_size = 2)(x)
    x = AddSingleLayer(x, filters=filters * 2)
    
    x = keras.layers.GlobalAveragePooling2D()(x)
    #x = layers.Flatten()(x)
    x = layers.Dense(layerSize, activation='softmax', name = lastLayerName, kernel_regularizer=cl2_reg)(x)
    return x

def create_resnet(input_shape):
    inputs = tf.keras.Input(shape=input_shape, dtype='float32', name='posts')
    common = layers.Conv2D(64, (7, 7), strides=(2, 2), padding='same')(inputs)
    common = layers.BatchNormalization()(common)
    common = layers.Activation('relu')(common)
    common = CommonBranchBlock(common)
    
    cho = BranchBlock(common,128,len(ja2label),'DenseCho2')
    jung = BranchBlock(common,128,len(mo2label),'DenseJung2')
    jong = BranchBlock(common,128,len(ba2label),'DenseJong2')
    
    model = tf.keras.Model(inputs, [cho, jung, jong])
    return model


In [11]:
model = create_resnet((64,64,3))
model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adamw', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

In [12]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ posts (InputLayer)  │ (None, 64, 64, 3) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      9,472 │ posts[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ activation[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     73,856 │ activation[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │    147,584 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    295,168 │ activation_3[0][… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │      1,024 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 256)              │            │                 

 Total params: 8,454,480 (32.25 MB)

 Trainable params: 8,446,672 (32.22 MB)

 Non-trainable params: 7,808 (30.50 KB)

In [14]:
model.load_weights(WEIGHT_FILE)

/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:396: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 206 variables. 
  trackable.load_own_variables(weights_store.get(inner_path))


In [13]:
save_dir = SAVE_DIR
checkPoint_path = WEIGHT_FILE

#from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
# 3번 반복내에 validation loss가 줄어들지 않으면 learning rate를 0.2 감소
#lr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, mode='min', verbose=1)
# 5번 반복내에 validation loss가 줄어들지 않으면 강제종료
#st_cb = EarlyStopping(monitor='val_loss', patience=5, mode='min', verbose=1)

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')
1302080
#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
model.fit(dataset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

Epoch 1/100


I0000 00:00:1717989378.317571   98886 service.cc:145] XLA service 0x7ff5100471f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1717989378.317612   98886 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-06-10 12:16:18.632505: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-06-10 12:16:19.638299: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


      1/Unknown 27s 27s/step - DenseCho2_accuracy: 0.0000e+00 - DenseJong2_accuracy: 0.0000e+00 - DenseJung2_accuracy: 0.0000e+00 - loss: 37.3119

I0000 00:00:1717989390.778334   98886 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  20004/Unknown 658s 32ms/step - DenseCho2_accuracy: 0.0561 - DenseJong2_accuracy: 0.1559 - DenseJung2_accuracy: 0.0711 - loss: 11.0905

...

20004/20004 ━━━━━━━━━━━━━━━━━━━━ 678s 34ms/step - DenseCho2_accuracy: 0.5273 - DenseJong2_accuracy: 0.5726 - DenseJung2_accuracy: 0.3855 - loss: 5.3869 - val_DenseCho2_accuracy: 0.1454 - val_DenseJong2_accuracy: 0.1561 - val_DenseJung2_accuracy: 0.1731 - val_loss: 9.7050
Epoch 22/100


2024-06-10 16:12:33.424311: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-06-10 16:12:33.424362: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2024-06-10 16:12:33.424395: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 16888671846052307659


20003/20004 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - DenseCho2_accuracy: 0.5270 - DenseJong2_accuracy: 0.5875 - DenseJung2_accuracy: 0.3991 - loss: 5.3621

2024-06-10 16:23:35.190312: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-06-10 16:23:35.190419: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_8]]
2024-06-10 16:24:03.240089: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-06-10 16:24:03.240140: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-06-10 16:24:03.240186: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 5344435933723410543
2024-06-10 16:24:03.240201: I tensorflow/core/framework/local_ren

20004/20004 ━━━━━━━━━━━━━━━━━━━━ 690s 34ms/step - DenseCho2_accuracy: 0.5270 - DenseJong2_accuracy: 0.5875 - DenseJung2_accuracy: 0.3991 - loss: 5.3621 - val_DenseCho2_accuracy: 0.1429 - val_DenseJong2_accuracy: 0.1049 - val_DenseJung2_accuracy: 0.1551 - val_loss: 9.9799
Epoch 23/100
10933/20004 ━━━━━━━━━━━━━━━━━━━━ 5:00 33ms/step - DenseCho2_accuracy: 0.5427 - DenseJong2_accuracy: 0.5953 - DenseJung2_accuracy: 0.3919 - loss: 5.3034

In [15]:
#model.save("./testModel.h5")
model.load_weights(WEIGHT_FILE)
model.save('./remodel3.keras')
